In [1]:
!nvidia-smi
!nvidia-smi --query-gpu=name,compute_cap --format=csv

Fri Nov  6 10:09:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!nvcc -arch=sm_75 ex1_timed.cu -o ex1_timed && ./ex1_timed

cc1plus: fatal error: ex1_timed.cu: No such file or directory
compilation terminated.


In [3]:
!ls -l *.cu

ls: cannot access '*.cu': No such file or directory


In [4]:
%%writefile ex1_timed.cu

Writing ex1_timed.cu


In [5]:
!nvcc -arch=sm_75 ex1_timed.cu -o ex1_timed && ./ex1_timed

/usr/bin/ld: /usr/lib/gcc/x86_64-linux-gnu/13/../../../x86_64-linux-gnu/Scrt1.o: in function `_start':
(.text+0x1b): undefined reference to `main'
collect2: error: ld returned 1 exit status


In [6]:
!rm -f ex1_timed.cu ex1_timed

In [11]:
%%writefile ex1_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;

    int size = N * sizeof(int);

    a = (int *)malloc(size);
    b = (int *)malloc(size);
    c = (int *)malloc(size);

    for (int i = 0; i < N; i++) {
        a[i] = rand() % 100;
        b[i] = rand() % 100;
    }

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    double start = cpuSecond();

    add<<<N, 1>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();

    double elapsed = cpuSecond() - start;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int i = 0; i < 5; i++)
        printf("%d + %d = %d\n", a[i], b[i], c[i]);

    int errors = 0;

    for (int i = 0; i < N; i++) {
        if (c[i] != a[i] + b[i])
            errors++;
    }

    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", elapsed);

    free(a);
    free(b);
    free(c);

    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);

    return 0;
}

Overwriting ex1_timed.cu


In [12]:
!nvcc -arch=sm_75 ex1_timed.cu -o ex1_timed && ./ex1_timed

83 + 86 = 169
77 + 15 = 92
93 + 35 = 128
86 + 92 = 178
49 + 21 = 70
Verification: 0 mismatches out of 512
Time taken - 0.0001221


In [13]:
!grep -n "cudaDeviceSynchronize" ex1_timed.cu

42:    cudaDeviceSynchronize();
